In [1]:
import os
import cv2
import datetime
import os
import time
import warnings
from collections import defaultdict, deque
import torch
import torch.utils.data
import torchvision
import torchvision.datasets.video_utils
from torch.utils.data.dataloader import default_collate
from torchvision.datasets.samplers import DistributedSampler, RandomClipSampler, UniformClipSampler
from collections.abc import Sequence
from functools import partial
from typing import Any, Callable, Optional, Union
from tqdm import tqdm
import torch
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.distributed as dist
from torchvision import transforms as T
from torchvision.models.video import R2Plus1D_18_Weights

In [2]:
class BasicBlock(nn.Module):

    expansion = 1   # output channels = expansion * planes (so our output channels = planes)

    def __init__(
        self,
        inplanes: int,  # input channels
        planes: int,    # output channels
        conv_builder: Callable[..., nn.Module], #generic convolution builder function
        stride: int = 1,    # transform features, but without shrinking
        downsample: Optional[nn.Module] = None, # adjusts skip connections
    ) -> None:
        midplanes = (inplanes * planes * 3 * 3 * 3) // (inplanes * 3 * 3 + 3 * planes)

        super().__init__()
        self.conv1 = nn.Sequential(
            conv_builder(inplanes, planes, midplanes, stride), nn.BatchNorm3d(planes), nn.ReLU(inplace=True)
        )   # convolutional layer with batch normalization and ReLU activation
        self.conv2 = nn.Sequential(conv_builder(planes, planes, midplanes), nn.BatchNorm3d(planes)) # convolutional layer with batch normalization (no activation)
        self.relu = nn.ReLU(inplace=True) # stops calculation from being a single linear transformation, inplace=True means it modifies the original so no new memory is needed to make a new one
        self.downsample = downsample
        self.stride = stride

    def forward(self, x: Tensor) -> Tensor:
        residual = x    # starting values

        out = self.conv1(x) #
        out = self.conv2(out)
        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out

In [3]:
class Conv2Plus1D(nn.Sequential):
    def __init__(self, in_planes: int, out_planes: int, midplanes: int, stride: int = 1, padding: int = 1) -> None:
        super().__init__(
            # Spatial conv first
            nn.Conv3d(
                in_planes,
                midplanes,
                kernel_size=(1, 3, 3),
                stride=(1, stride, stride),
                padding=(0, padding, padding),
                bias=False,
            ),
            nn.BatchNorm3d(midplanes),  # stabilizes training by stopping some activations from crashing back to zero
            nn.ReLU(inplace=True), # stops calculation from being a single linear transformation, inplace=True means it modifies the original so no new memory is needed to make a new one
            # Temporal conv second
            nn.Conv3d(
                midplanes, out_planes, kernel_size=(3, 1, 1), stride=(stride, 1, 1), padding=(padding, 0, 0), bias=False
            ),
        )

    @staticmethod
    def get_downsample_stride(stride: int) -> tuple[int, int, int]:
        return stride, stride, stride

In [4]:
class R2Plus1dStem(nn.Sequential):
    """R(2+1)D stem is different than the default one as it uses separated 3D convolution"""
    
    # nn.Conv3d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1,
    # groups=1, bias=True, padding_mode='zeros', device=None, dtype=None)
    # or rather
    # Input: (N, Cin, T, H, W)
    # Output: (N, Cout, Tout, Hout, Wout)
    
    # nn.BatchNorm3d(num_features, eps=1e-05, momentum=0.1, affine=True,
    # track_running_stats=True, device=None, dtype=None)
    # or rather
    # Input: (N, C, T, H, W)
    # Output: (N, C, T, H, W) (same shape)
    
    def __init__(self) -> None:
        super().__init__(
            # Spatial convolution
            nn.Conv3d(3, 45, kernel_size=(1, 7, 7), stride=(1, 2, 2), padding=(0, 3, 3), bias=False),
            nn.BatchNorm3d(45),
            nn.ReLU(inplace=True),
            # Temporal convolution
            nn.Conv3d(45, 64, kernel_size=(3, 1, 1), stride=(1, 1, 1), padding=(1, 0, 0), bias=False),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
        )

In [5]:
class VideoResNet(nn.Module):
    def __init__(
        self,
        block: type[BasicBlock],
        conv_makers: type[Conv2Plus1D],
        layers: list[int],
        stem: Callable[..., nn.Module], #generic callable stem function
        weights: Optional[R2Plus1D_18_Weights] = None,
        progress: bool = True,
        num_classes: int = 400,
        **kwargs: Any,
    ) -> None:
        """Generic resnet video generator.

        Args:
            block (Type[Union[BasicBlock, Bottleneck]]): resnet building block
            conv_makers (List[Type[Union[Conv3DSimple, Conv3DNoTemporal, Conv2Plus1D]]]): generator
                function for each layer
            layers (List[int]): number of blocks per layer
            stem (Callable[..., nn.Module]): module specifying the ResNet stem.
            weights (Optional[R2Plus1D_18_Weights]): pretrained weights. None = train from scratch.
            progress (bool): If True, show download progress bar. Defaults to True.
            num_classes (int, optional): Dimension of the final FC layer. Defaults to 400.
        """
        super().__init__()
        self.inplanes = 64

        self.stem = stem()
                        # BasicBlock, Conv2Plus1D, output channel size, layers[] gives blocks per layer
        self.layer1 = self._make_layer(block, conv_makers, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(block, conv_makers, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, conv_makers, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, conv_makers, 512, layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        # init weights
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

        # Load pretrained weights if provided
        if weights is not None:
            self.load_state_dict(weights.get_state_dict(progress=progress, check_hash=True))

    def load_pretrained(self, weights: R2Plus1D_18_Weights, progress: bool = True) -> None:
        """Load pretrained weights after initialization (e.g. for inference).

        Args:
            weights: The pretrained weights to load (e.g. R2Plus1D_18_Weights.DEFAULT).
            progress: If True, show download progress bar.
        """
        self.load_state_dict(weights.get_state_dict(progress=progress, check_hash=True))

    def forward(self, x: Tensor) -> Tensor:
        x = self.stem(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        # Flatten the layer to fc
        x = x.flatten(1)
        x = self.fc(x)

        return x

    def _make_layer(
        self,
        block: type[BasicBlock],
        conv_builder: type[Conv2Plus1D],
        planes: int,    # output channels
        blocks: int,
        stride: int = 1,
    ) -> nn.Sequential:
        downsample = None

        if stride != 1 or self.inplanes != planes * block.expansion:
            ds_stride = conv_builder.get_downsample_stride(stride)
            downsample = nn.Sequential(
                nn.Conv3d(self.inplanes, planes * block.expansion, kernel_size=1, stride=ds_stride, bias=False),
                nn.BatchNorm3d(planes * block.expansion),
            )
        layers = []
        layers.append(block(self.inplanes, planes, conv_builder, stride, downsample))

        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes, conv_builder))

        return nn.Sequential(*layers)

In [6]:
# Train from scratch (no pretrained weights)
model = VideoResNet(
    BasicBlock,
    Conv2Plus1D,
    [2, 2, 2, 2],
    R2Plus1dStem,
    num_classes=5,  # 5 pitch types for Woo
)

In [7]:
class VideoDataset(Dataset):
    def __init__(
        self,
        video_paths,
        labels,
        num_frames=180,
        resize=(1280, 720)
        ):
        
        self.video_paths = video_paths
        self.labels = labels
        self.num_frames = num_frames
        self.resize = resize
        self.transform = T.Compose([T.ToTensor(),
            # Y'know how a lot of AI Images have that yellow tint?
            # We're normalizing to prevent that and speed up training
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        cap = cv2.VideoCapture(self.video_paths[idx])
        frames = []
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        # Calculate stride to sample exactly 'num_frames' across
        # the video
        stride = max(1, total_frames // self.num_frames)
        for i in range(self.num_frames):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i * stride)
            ret, frame = cap.read()
            if not ret:
                # If video ends early, pad with a blank frame
                frame = torch.zeros((3, *self.resize))
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, self.resize)
                frame = self.transform(frame)
            frames.append(frame)
        cap.release()
        # Stack frames into [Channels, Frames, H, W]
        video_tensor = torch.stack(frames, dim=1)
        return video_tensor, self.labels[idx]

In [8]:
def get_paths_and_labels(root_dir):
    video_paths = []
    labels = []
    class_to_idx = {
        cls: i for i, cls in enumerate(sorted(os.listdir(root_dir)))
    }
    for cls in class_to_idx:
        cls_dir = os.path.join(root_dir, cls)
        for vid in os.listdir(cls_dir):
            video_paths.append(os.path.join(cls_dir, vid))
            labels.append(class_to_idx[cls])
    return video_paths, labels, class_to_idx

In [9]:
# ---- Data ----
train_paths, train_labels, class_map = get_paths_and_labels(
    'D:/Github/BaseballPitch/modules/naive_classification/train'
)
val_paths, val_labels, _ = get_paths_and_labels(
    'D:/Github/BaseballPitch/modules/naive_classification/val'
)

train_dataset = VideoDataset(train_paths, train_labels)
val_dataset = VideoDataset(val_paths, val_labels)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    prefetch_factor=4,
    persistent_workers=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    prefetch_factor=4,
    persistent_workers=True
)

In [10]:
print("Class mapping:", class_map)
print("Unique labels in train_labels:", set(train_labels))
print("Number of videos per class:")
from collections import Counter
print(Counter(train_labels))
print("Folders found:", list(class_map.keys()))

Class mapping: {'CH - Changeup': 0, 'FF - Fastball': 1, 'SI - Sinker': 2, 'SL - Slider': 3, 'ST - Sweeper': 4}
Unique labels in train_labels: {0, 1, 2, 3, 4}
Number of videos per class:
Counter({1: 2165, 2: 1174, 3: 567, 4: 420, 0: 344})
Folders found: ['CH - Changeup', 'FF - Fastball', 'SI - Sinker', 'SL - Slider', 'ST - Sweeper']


In [11]:
class SmoothedValue:
    """Track a series of values and provide access to smoothed values over a
    window or the global series average.
    """

    def __init__(self, window_size=20, fmt=None):
        if fmt is None:
            fmt = "{median:.4f} ({global_avg:.4f})"
        self.deque = deque(maxlen=window_size)
        self.total = 0.0
        self.count = 0
        self.fmt = fmt

    def update(self, value, n=1):
        self.deque.append(value)
        self.count += n
        self.total += value * n

    def synchronize_between_processes(self):
        """
        Warning: does not synchronize the deque!
        """
        t = reduce_across_processes([self.count, self.total])
        t = t.tolist()
        self.count = int(t[0])
        self.total = t[1]

    @property
    def median(self):
        d = torch.tensor(list(self.deque))
        return d.median().item()

    @property
    def avg(self):
        d = torch.tensor(list(self.deque), dtype=torch.float32)
        return d.mean().item()

    @property
    def global_avg(self):
        return self.total / self.count

    @property
    def max(self):
        return max(self.deque)

    @property
    def value(self):
        return self.deque[-1]

    def __str__(self):
        return self.fmt.format(
            median=self.median, avg=self.avg, global_avg=self.global_avg, max=self.max, value=self.value
        )


class MetricLogger:
    def __init__(self, delimiter="\t"):
        self.meters = defaultdict(SmoothedValue)
        self.delimiter = delimiter

    def update(self, **kwargs):
        for k, v in kwargs.items():
            if isinstance(v, torch.Tensor):
                v = v.item()
            if not isinstance(v, (float, int)):
                raise TypeError(
                    f"This method expects the value of the input arguments to be of type float or int, instead  got {type(v)}"
                )
            self.meters[k].update(v)

    def __getattr__(self, attr):
        if attr in self.meters:
            return self.meters[attr]
        if attr in self.__dict__:
            return self.__dict__[attr]
        raise AttributeError(f"'{type(self).__name__}' object has no attribute '{attr}'")

    def __str__(self):
        loss_str = []
        for name, meter in self.meters.items():
            loss_str.append(f"{name}: {str(meter)}")
        return self.delimiter.join(loss_str)

    def synchronize_between_processes(self):
        for meter in self.meters.values():
            meter.synchronize_between_processes()

    def add_meter(self, name, meter):
        self.meters[name] = meter

    def log_every(self, iterable, print_freq, header=None):
        i = 0
        if not header:
            header = ""
        start_time = time.time()
        end = time.time()
        iter_time = SmoothedValue(fmt="{avg:.4f}")
        data_time = SmoothedValue(fmt="{avg:.4f}")
        space_fmt = ":" + str(len(str(len(iterable)))) + "d"
        if torch.cuda.is_available():
            log_msg = self.delimiter.join(
                [
                    header,
                    "[{0" + space_fmt + "}/{1}]",
                    "eta: {eta}",
                    "{meters}",
                    "time: {time}",
                    "data: {data}",
                    "max mem: {memory:.0f}",
                ]
            )
        else:
            log_msg = self.delimiter.join(
                [header, "[{0" + space_fmt + "}/{1}]", "eta: {eta}", "{meters}", "time: {time}", "data: {data}"]
            )
        MB = 1024.0 * 1024.0
        for obj in iterable:
            data_time.update(time.time() - end)
            yield obj
            iter_time.update(time.time() - end)
            if i % print_freq == 0:
                eta_seconds = iter_time.global_avg * (len(iterable) - i)
                eta_string = str(datetime.timedelta(seconds=int(eta_seconds)))
                if torch.cuda.is_available():
                    print(
                        log_msg.format(
                            i,
                            len(iterable),
                            eta=eta_string,
                            meters=str(self),
                            time=str(iter_time),
                            data=str(data_time),
                            memory=torch.cuda.max_memory_allocated() / MB,
                        )
                    )
                else:
                    print(
                        log_msg.format(
                            i, len(iterable), eta=eta_string, meters=str(self), time=str(iter_time), data=str(data_time)
                        )
                    )
            i += 1
            end = time.time()
        total_time = time.time() - start_time
        total_time_str = str(datetime.timedelta(seconds=int(total_time)))
        print(f"{header} Total time: {total_time_str}")

def accuracy(output, target, topk=(1,)):
    """Computes the accuracy over the k top predictions for the specified values of k"""
    with torch.inference_mode():
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target[None])

        res = []
        for k in topk:
            correct_k = correct[:k].flatten().sum(dtype=torch.float32)
            res.append(correct_k * (100.0 / batch_size))
        return res

def is_dist_avail_and_initialized():
    if not dist.is_available():
        return False
    if not dist.is_initialized():
        return False
    return True

def reduce_across_processes(val, op=dist.ReduceOp.SUM):
    if not is_dist_avail_and_initialized():
        # nothing to sync, but we still convert to tensor for consistency with the distributed case.
        return torch.tensor(val)

    t = torch.tensor(val, device="cuda")
    dist.barrier()
    dist.all_reduce(t, op=op)
    return t

In [12]:
def train_one_epoch(model, criterion, optimizer, lr_scheduler, data_loader, device, epoch, print_freq, scaler=None):
    model.train()
    metric_logger = MetricLogger(delimiter="  ")
    metric_logger.add_meter("lr", SmoothedValue(window_size=1, fmt="{value}"))
    metric_logger.add_meter("clips/s", SmoothedValue(window_size=10, fmt="{value:.3f}"))

    header = f"Epoch: [{epoch}]"
    for video, target, _ in metric_logger.log_every(data_loader, print_freq, header):
        start_time = time.time()
        video, target = video.to(device), target.to(device)
        with torch.cuda.amp.autocast(enabled=scaler is not None):
            output = model(video)
            loss = criterion(output, target)

        optimizer.zero_grad()

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        acc1, acc5 = accuracy(output, target, topk=(1, 5))
        batch_size = video.shape[0]
        metric_logger.update(loss=loss.item(), lr=optimizer.param_groups[0]["lr"])
        metric_logger.meters["acc1"].update(acc1.item(), n=batch_size)
        metric_logger.meters["acc5"].update(acc5.item(), n=batch_size)
        metric_logger.meters["clips/s"].update(batch_size / (time.time() - start_time))
        lr_scheduler.step()

In [13]:
def evaluate(model, criterion, data_loader, device):
    model.eval()
    metric_logger = MetricLogger(delimiter="  ")
    header = "Test:"
    num_processed_samples = 0
    # Group and aggregate output of a video
    num_videos = len(data_loader.dataset.samples)
    num_classes = len(data_loader.dataset.classes)
    agg_preds = torch.zeros((num_videos, num_classes), dtype=torch.float32, device=device)
    agg_targets = torch.zeros((num_videos), dtype=torch.int32, device=device)
    with torch.inference_mode():
        for video, target, video_idx in metric_logger.log_every(data_loader, 100, header):
            video = video.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            output = model(video)
            loss = criterion(output, target)

            # Use softmax to convert output into prediction probability
            preds = torch.softmax(output, dim=1)
            for b in range(video.size(0)):
                idx = video_idx[b].item()
                agg_preds[idx] += preds[b].detach()
                agg_targets[idx] = target[b].detach().item()

            acc1, acc5 = accuracy(output, target, topk=(1, 5))
            # FIXME need to take into account that the datasets
            # could have been padded in distributed setup
            batch_size = video.shape[0]
            metric_logger.update(loss=loss.item())
            metric_logger.meters["acc1"].update(acc1.item(), n=batch_size)
            metric_logger.meters["acc5"].update(acc5.item(), n=batch_size)
            num_processed_samples += batch_size
    # gather the stats from all processes
    num_processed_samples = reduce_across_processes(num_processed_samples)
    if isinstance(data_loader.sampler, DistributedSampler):
        # Get the len of UniformClipSampler inside DistributedSampler
        num_data_from_sampler = len(data_loader.sampler.dataset)
    else:
        num_data_from_sampler = len(data_loader.sampler)

    if (
        hasattr(data_loader.dataset, "__len__")
        and num_data_from_sampler != num_processed_samples
        and torch.distributed.get_rank() == 0
    ):
        # See FIXME above
        warnings.warn(
            f"It looks like the sampler has {num_data_from_sampler} samples, but {num_processed_samples} "
            "samples were used for the validation, which might bias the results. "
            "Try adjusting the batch size and / or the world size. "
            "Setting the world size to 1 is always a safe bet."
        )

    metric_logger.synchronize_between_processes()

    print(
        " * Clip Acc@1 {top1.global_avg:.3f} Clip Acc@5 {top5.global_avg:.3f}".format(
            top1=metric_logger.acc1, top5=metric_logger.acc5
        )
    )
    # Reduce the agg_preds and agg_targets from all gpu and show result
    agg_preds = reduce_across_processes(agg_preds)
    agg_targets = reduce_across_processes(agg_targets, op=torch.distributed.ReduceOp.MAX)
    agg_acc1, agg_acc5 = accuracy(agg_preds, agg_targets, topk=(1, 5))
    print(" * Video Acc@1 {acc1:.3f} Video Acc@5 {acc5:.3f}".format(acc1=agg_acc1, acc5=agg_acc5))
    return metric_logger.acc1.global_avg

In [14]:
# ---- GPU Setup ----
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. GPU required.")
device = torch.device("cuda")
torch.cuda.set_device(0)
torch.backends.cudnn.benchmark = True
print(f"Using GPU: {torch.cuda.get_device_name(0)}")

# ---- Model ----
model = model.to(device)

# Verify model is on GPU
print(f"Model device: {next(model.parameters()).device}")

# ---- Optimizer & Loss ----
optimizer = torch.optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(), 'lr': 1e-3},
])
criterion = nn.CrossEntropyLoss().to(device)

Using GPU: NVIDIA GeForce RTX 2070 SUPER
Model device: cuda:0


In [ ]:
# ---- Training Loop ----
num_epochs = 10
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs * len(train_loader))
scaler = torch.cuda.amp.GradScaler()
checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

for epoch in range(num_epochs):
    # --- Train ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    metric_logger = MetricLogger(delimiter="  ")
    metric_logger.add_meter("lr", SmoothedValue(window_size=1, fmt="{value:.6f}"))
    metric_logger.add_meter("clips/s", SmoothedValue(window_size=10, fmt="{value:.3f}"))

    for batch_idx, (video, target) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")):
        start_time = time.time()
        video, target = video.to(device), target.to(device)

        with torch.cuda.amp.autocast():
            output = model(video)
            loss = criterion(output, target)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()

        acc1, acc5 = accuracy(output, target, topk=(1, min(5, 5)))
        batch_size = video.size(0)
        running_loss += loss.item() * batch_size
        _, predicted = output.max(1)
        total += batch_size
        correct += predicted.eq(target).sum().item()

        # Update MetricLogger
        metric_logger.update(loss=loss.item(), lr=optimizer.param_groups[0]["lr"])
        metric_logger.meters["acc1"].update(acc1.item(), n=batch_size)
        metric_logger.meters["acc5"].update(acc5.item(), n=batch_size)
        metric_logger.meters["clips/s"].update(batch_size / (time.time() - start_time))

    train_loss = running_loss / total
    train_acc = 100. * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f} Train Acc: {train_acc:.2f}%")
    print(f"  Metrics: {metric_logger}")

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    val_metric_logger = MetricLogger(delimiter="  ")

    with torch.no_grad():
        for video, target in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
            video, target = video.to(device), target.to(device)

            with torch.cuda.amp.autocast():
                output = model(video)
                loss = criterion(output, target)

            batch_size = video.size(0)
            val_loss += loss.item() * batch_size
            _, predicted = output.max(1)
            val_total += batch_size
            val_correct += predicted.eq(target).sum().item()

            acc1, acc5 = accuracy(output, target, topk=(1, min(5, 5)))
            val_metric_logger.update(loss=loss.item())
            val_metric_logger.meters["acc1"].update(acc1.item(), n=batch_size)
            val_metric_logger.meters["acc5"].update(acc5.item(), n=batch_size)

    val_loss /= val_total
    val_acc = 100. * val_correct / val_total
    print(f"Epoch [{epoch+1}/{num_epochs}] Val Loss: {val_loss:.4f} Val Acc: {val_acc:.2f}%")
    print(f"  Metrics: {val_metric_logger}")
    print("-" * 60)

    # --- Save Checkpoint ---
    checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch+1:02d}.pth")
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "lr_scheduler_state_dict": lr_scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    }, checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}")

# Save the trained model
torch.save(model.state_dict(), "r2plus1d_pitch_classifier.pth")
print("Model saved to r2plus1d_pitch_classifier.pth")

C:\Users\reill\AppData\Local\Temp\ipykernel_22620\2038240620.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/10 [Train]:   0%|          | 0/292 [00:00<?, ?it/s]